In [ ]:
import os
import gradio as gr
from langchain_community.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_openai import ChatOpenAI
from langchain.chains import RetrievalQA
from langchain.prompts import PromptTemplate

# --- CONFIGURATION ---
# Replace with your actual OpenAI API key
os.environ["OPENAI_API_KEY"] = "YOUR_OPENAI_API_KEY_HERE"

# --- STEP 1: LOAD AND PROCESS DATA  ---
# Using the specific filename provided
pdf_filename = "the_nestle_hr_policy_pdf_2012.pdf" 

# Check if file exists to prevent errors
if not os.path.exists(pdf_filename):
    raise FileNotFoundError(f"The file '{pdf_filename}' was not found in the directory.")

loader = PyPDFLoader(pdf_filename)
documents = loader.load()

# Split text into chunks for vectorization
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)
texts = text_splitter.split_documents(documents)
print(f"Successfully loaded and split {len(texts)} text chunks.")

# --- STEP 2: CREATE VECTOR STORE  ---
# Initialize OpenAI embeddings and Chroma DB
embeddings = OpenAIEmbeddings()
db = Chroma.from_documents(
    documents=texts, 
    embedding=embeddings
)
print("Vector store created successfully.")

# --- STEP 3: BUILD QA SYSTEM [cite: 24] ---
# Define the prompt template [cite: 25]
prompt_template = """You are a helpful HR Assistant for Nestlé. 
Use the following pieces of context to answer the question regarding HR policies. 
If the answer is not in the context, state that you do not know.

Context: {context}

Question: {question}
Answer:"""

PROMPT = PromptTemplate(
    template=prompt_template, 
    input_variables=["context", "question"]
)

# Initialize GPT-3.5 Turbo model
llm = ChatOpenAI(model_name="gpt-3.5-turbo", temperature=0)

# Create the Retrieval Chain
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=db.as_retriever(search_kwargs={"k": 3}),
    return_source_documents=True,
    chain_type_kwargs={"prompt": PROMPT}
)

# --- STEP 4: GRADIO INTERFACE  ---
def hr_bot(question):
    # Retrieve answer from the chain
    response = qa_chain.invoke({"query": question})
    return response["result"]

# Build the UI
iface = gr.Interface(
    fn=hr_bot,
    inputs=gr.Textbox(lines=2, placeholder="Ask about Nestlé HR policies..."),
    outputs="text",
    title="Nestlé HR Policy Assistant",
    description="AI-powered chatbot for answering queries on Nestlé's HR documents."
)

# Launch the application
iface.launch(share=True)